# Collect xDiyo seasons manually

This notebook runs the new scraper in the notebook's own Python process. It collects the seasons listed in `data/xDiyo_data_promotions` into `data/xDiyo_data`, keeping the filename stems and using Parquet. Previously saved responses are reused.

**Quick start:** run the setup and inspection cells, configure Proton VPN yourself, then set `RUN_COLLECTION = True` in **Start or resume** and run that cell. The initial `False` setting lets you run the entire notebook to inspect the handoff without contacting Sofascore. For a CAPTCHA, set `MANUAL_CAPTCHA = True` in the configuration cell: the scraper opens Sofascore and waits for you before retrying the saved block. For other resolved access problems, use `RETRY_BLOCKED_ONCE = True`.

The configured collection includes xG, lineups, heatmaps, pregame standings, injury availability and supplied goal buildup actions. Commentary is disabled. Requests run serially with a five-minute pause between seasons. Provider cooldowns still apply.

## 1. Setup

In VS Code, select this project's `.venv/Scripts/python.exe` as the notebook kernel. Dependencies have been installed in that environment. To launch JupyterLab yourself, run this from the repository root in PowerShell:

```powershell
.\.venv\Scripts\python.exe -m jupyterlab "scraping/Scrape+wait/collect_historical_seasons.ipynb"
```

For a fresh environment, install the notebook dependencies first:

```powershell
.\.venv\Scripts\python.exe -m pip install -e '.[notebook]'
```

Keep the notebook kernel and computer running during collection. Use **Interrupt Kernel** to stop; completed responses remain saved. After editing the scraper package, restart the kernel before using the changed code.

In [ ]:
from collections import Counter
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import sys
import time

# Set this explicitly if you move the notebook outside the project.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / "scraping/sofascore/campaign.py").is_file()),
        None,
    )
if PROJECT_ROOT is None:
    raise ValueError("Set PROJECT_ROOT to the xDiyo repository directory.")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import clear_output, display
from scraping.sofascore import campaign
from scraping.sofascore.models import CollectionPaused
from scraping.sofascore.planner import component_jobs
from scraping.sofascore.store import Store

print(f"Project: {PROJECT_ROOT}")
print(f"Kernel:  {sys.executable}")

## 2. Choose seasons, pacing and ChromeDriver

Leave `SEASONS = None` to collect all 110 legacy filename stems, newest seasons first. To run a smaller selection, use exact stems such as `['Bundesliga_24_25']`.

Leave `DRIVER_PATH = None` for Selenium to find/download a driver compatible with installed Chrome when collection starts. Alternatively, provide an absolute path to a matching `chromedriver.exe`. Updating Chrome can require a different driver. The browser uses your current external VPN connection.

Keep `STORE` unchanged when resuming: it holds the saved API responses and shared request ledger. The source CSVs define season and round scope; their old values are not substituted for new API responses. The known mislabeled Eredivisie 2016/17 source is resolved against the provider's season list when reached.

Set `RANDOM_DELAY_MAX_SECONDS = T` below to add a fresh uniform random pause from 0 to `T` seconds between outgoing requests. `None` (the default) disables this delay; `0` also adds no delay. It applies to retries, skips cached data, and adds to other required pacing. The first request in a fresh store has no added delay. Negative or non-finite values are rejected.

**Manual CAPTCHA:** set `MANUAL_CAPTCHA = True` to open the regular Sofascore homepage in the scraper's visible Chrome session. Complete the challenge yourself, return to the notebook, and press Enter at its prompt. The same session is used for subsequent requests and seasons. This confirmation permits a retry; the next API response determines whether access is restored. Homepage traffic runs normally and is outside the per-request delay control; observed API calls are recorded separately.

In [ ]:
SOURCE = PROJECT_ROOT / "data/xDiyo_data_promotions"
OUTPUT = PROJECT_ROOT / "data/xDiyo_data"
STORE = PROJECT_ROOT / "data/collection_state"
CONTROL = OUTPUT / "_collection"

SEASONS = None  # Or, for example: ["Bundesliga_24_25"]
DRIVER_PATH = None  # Or Path(r"C:\path\to\chromedriver.exe")
RANDOM_DELAY_MAX_SECONDS = None  # Example: 2.0 adds a random 0-2 second pause.
MANUAL_CAPTCHA = False  # Open Sofascore and wait for you to complete the challenge.

assert SOURCE.is_dir(), f"Source directory not found: {SOURCE}"
if SEASONS is not None and (not isinstance(SEASONS, list) or not SEASONS):
    raise ValueError("Use None for all seasons or a nonempty list of exact stems.")
if DRIVER_PATH is not None and not Path(DRIVER_PATH).is_file():
    raise ValueError(f"ChromeDriver not found: {DRIVER_PATH}")

def collection_arguments():
    arguments = ["--source", str(SOURCE), "--output", str(OUTPUT), "--store", str(STORE)]
    if DRIVER_PATH is not None:
        arguments += ["--driver-path", str(DRIVER_PATH)]
    if RANDOM_DELAY_MAX_SECONDS is not None:
        arguments += ["--random-delay-max-seconds", str(RANDOM_DELAY_MAX_SECONDS)]
    if MANUAL_CAPTCHA:
        arguments.append("--manual-captcha")
    for season in SEASONS or []:
        arguments += ["--season", season]
    return arguments

display(pd.DataFrame([
    {"setting": key, "value": value}
    for key, value in asdict(campaign.PROFILE).items()
    if key.startswith("include_") or key == "duration_mode"
]))
print("Cadence: serial requests; 300 seconds between seasons.")
print("No hourly/run cap; Retry-After and access blocks remain enforced.")
print("Random delay: disabled" if RANDOM_DELAY_MAX_SECONDS is None else
      f"Random delay: 0 to {RANDOM_DELAY_MAX_SECONDS} seconds between requests")

## 3. Inspect the season queue — no API requests

This validates the filename selection and prepares or reuses the inventory. Only fully exported seasons receive a `.manifest.json`; partially collected seasons stay in the shared cache.

In [ ]:
campaign.main(collection_arguments() + ["--prepare-only"])
inventory = json.loads((CONTROL / "inventory.json").read_text(encoding="utf-8"))["seasons"]
selected = [item for item in inventory if SEASONS is None or item["stem"] in SEASONS]
queue = pd.DataFrame([
    {"season": item["stem"], "season_id": item["season_id"],
     "legacy_matches": item["legacy_rows"],
     "export_manifest_present": (OUTPUT / (item["stem"] + ".manifest.json")).is_file()}
    for item in selected
])
print(f"Selected: {len(selected)} of {len(inventory)} seasons")
display(queue.head(15))

## 4. Inspect the saved checkpoint — no API requests

The status file describes the last run. This cell recomputes component counts from the cache for that run's current season, so it can show progress more precisely than the periodically saved counter. `unavailable` means the API reported the component absent; it does not mean a blocked request was treated as missing data.

In [ ]:
def show_checkpoint():
    path = CONTROL / "status.json"
    status = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    print(f"Last run: {status.get('state', 'not started')} | "
          f"season: {status.get('current_season', 'none')} | "
          f"phase: {status.get('phase', 'none')}")
    if not (STORE / "manifest.sqlite").exists():
        print("No response cache yet.")
        return
    with Store(STORE) as store:
        block = store.setting("access_block")
        cooldown = store.setting("cooldown_until", 0)
        print(f"Saved access block: {block}")
        print(f"Remaining server cooldown: {max(0, cooldown - time.time()):.0f} seconds")
        current = next((item for item in inventory
                        if item["stem"] == status.get("current_season")), None)
        season_id = status.get("season_id")
        if current is not None and season_id is not None:
            events = store.events(current["league_id"], season_id)
            jobs = [job for event in events
                    for job in component_jobs(event, campaign.PROFILE, time.time())]
            counts = Counter((store.get(job) or {}).get("status", "not_requested") for job in jobs)
            terminal = sum(counts[name] for name in campaign.TERMINAL)
            print(f"Fixtures discovered: {len(events)} | components finished: {terminal}/{len(jobs)}")
            display(pd.DataFrame(sorted(counts.items()), columns=["component_status", "count"]))
    print(f"Stop marker present: {(CONTROL / 'STOP').exists()}")

show_checkpoint()

## 5. Start or resume

1. Configure your external VPN if needed. For a CAPTCHA, set `MANUAL_CAPTCHA = True` in the configuration cell above and rerun that cell.
2. Set `RUN_COLLECTION = True` below and run this cell.
3. If manual CAPTCHA is enabled, the scraper opens the normal Sofascore homepage. Complete the challenge in that window, let the website load, then return here and press Enter at the notebook prompt. Keep that browser window open while collecting.

Manual CAPTCHA confirmation also permits retrying the saved access block, so `RETRY_BLOCKED_ONCE` can stay `False`. That flag is still available for other access problems you have already resolved. An active server cooldown is honored before opening the website. **Interrupt Kernel** cancels; cancelling before confirmation preserves the saved block.

The scraper keeps this browser session across the selected seasons and their five-minute waits. After confirmation it moves to a blank page to stop homepage polling, then fetches the remaining components. The session closes when the cell finishes, pauses, or is interrupted. A repeat `403` still pauses collection; rerun with manual CAPTCHA enabled to try again yourself. A `429` preserves and waits for the provider cooldown. There is no automatic challenge solver or VPN control.

Successful cached responses are reused. A completed selection opens no browser. The old STOP marker is removed only after collection obtains its locks. A season is published only when its requested components are complete; generic errors retain their traceback for diagnosis.

The two controls below reset in memory after an attempted run. Setting them to `True` in this cell's source means rerunning the cell explicitly starts/retries again; change them back to `False` before saving if you want **Run All** to stay in inspection mode.

In [ ]:
RUN_COLLECTION = False
RETRY_BLOCKED_ONCE = False

def show_progress(status):
    clear_output(wait=True)
    timestamp = datetime.fromtimestamp(status["updated_at"], timezone.utc).isoformat(timespec="seconds")
    print(f"{timestamp} | {status['state']} | {status.get('phase', '')}")
    print(f"Season: {status.get('current_season', 'checking completed exports')} | "
          f"completed seasons: {status['completed_seasons']}/{status['total_seasons']}")
    if status.get("phase") in {"components", "exporting", "access_block"}:
        print(f"Saved component progress: {status.get('terminal_components', 0)}/"
              f"{status.get('requested_components', '?')}")
    if status.get("state") == "waiting":
        resume_at = datetime.fromtimestamp(status["resume_at"], timezone.utc).isoformat(timespec="seconds")
        print(f"Resume at {resume_at}")
    if status.get("error"):
        print(status["error"])

if RUN_COLLECTION:
    arguments = collection_arguments() + ["--resume", "--pause-on-block"]
    if RETRY_BLOCKED_ONCE:
        arguments.append("--retry-blocked")
    try:
        campaign.main(arguments, on_progress=show_progress)
        print("All selected seasons have completed exports.")
    except CollectionPaused as exc:
        print(f"Paused: {exc}")
        print("Completed responses are saved. Inspect the checkpoint before resuming.")
    except KeyboardInterrupt:
        print("Stopped by you. Completed responses are saved; rerun this cell to resume.")
    finally:
        RUN_COLLECTION = False
        RETRY_BLOCKED_ONCE = False
else:
    print("Inspection mode. Set RUN_COLLECTION = True above when ready to collect.")

## 6. Review progress and completed exports

Rerun this after collection pauses or finishes. The table reports completed season manifests; an interrupted first season will not appear yet. The next cell verifies the selected export's SHA-256 and previews scalar match columns without loading all nested heatmap points.

In [ ]:
show_checkpoint()
completed = []
for item in selected:
    manifest_path = OUTPUT / (item["stem"] + ".manifest.json")
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("complete"):
            completed.append({"season": item["stem"], "matches": manifest["matches"],
                              "shots_with_xg": manifest["shots_with_xg"],
                              "lineup_events": manifest["lineup_events"]})
print(f"Completed exports: {len(completed)}/{len(selected)} selected seasons")
display(pd.DataFrame(completed).head(20))

In [ ]:
INSPECT_SEASON = None  # Or an exact completed stem, e.g. "Bundesliga_24_25"
inspect_season = INSPECT_SEASON or (completed[0]["season"] if completed else None)
if inspect_season is None:
    print("No completed season export yet. The saved responses remain resumable.")
else:
    manifest = json.loads((OUTPUT / (inspect_season + ".manifest.json")).read_text(encoding="utf-8"))
    season_path = OUTPUT / manifest["season_file"]
    assert hashlib.sha256(season_path.read_bytes()).hexdigest() == manifest["sha256"], "File hash mismatch"
    parquet = pq.ParquetFile(season_path)
    assert parquet.metadata.num_rows == manifest["matches"], "Match count mismatch"
    columns = ["event_id", "kickoff_utc", "home_name", "away_name", "round", "status"]
    preview = parquet.read(columns=columns).to_pandas().head(10)
    preview["kickoff_utc"] = pd.to_datetime(preview["kickoff_utc"], unit="s", utc=True)
    print(f"Verified: {season_path.name} | {parquet.metadata.num_rows} matches")
    display(preview)
    print("Component coverage:")
    display(manifest["coverage"])

## 7. Optional: load the normalized legacy view

The stored Parquet keeps provider values. `merge_seasons` produces the existing normalized standings and round features: home and away standings are each divided by their observed season maximum, and rounds by their observed maximum. Missing standings remain missing. Lineups are included in the example.

Promotion/relegation flags are a separate local join. Set `TEAM_SEASON_FLAGS` to the prepared flags manifest to include them. The earlier flags inventory excludes the mislabeled Eredivisie 2016/17 file; recollecting that season alone does not rebuild that inventory. Unknown movements remain unknown.

Keep the `_tables` directory alongside the season Parquet files: the compatibility loader follows their relative manifest links. xG and goal buildup depend on provider coverage, historical injury availability is not proof of what was known before kickoff, and exact match/half durations remain null where the source does not establish them.

In [ ]:
LOAD_LEGACY_VIEW = False  # Loading a season's heatmaps can use substantial memory.
TEAM_SEASON_FLAGS = None
# TEAM_SEASON_FLAGS = PROJECT_ROOT / "data/prepared_v2/team_seasons/CURRENT.json"

if LOAD_LEGACY_VIEW and inspect_season is not None:
    from utils.data_loading import merge_seasons
    matches, league_indices = merge_seasons(
        [inspect_season], data_dir=OUTPUT, include_lineups=True,
        team_season_flags=TEAM_SEASON_FLAGS,
    )
    preview_columns = [name for name in
                       ["match_id", "home_standing", "away_standing", "normalized_round"]
                       if name in matches.columns]
    display(matches[preview_columns].head())
else:
    print("Legacy view not loaded. Enable it after a selected season completes.")